In [35]:
%reload_ext autoreload
%autoreload 2

import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import altair as alt
from preprocessing import *
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.stats import kendalltau
from final_views.q1_ratings_evolution import make_plot_q1
from final_views.q2_viewers_evolution import make_plot_q2
#from final_views.q3_correlation import make_plot_q3
from final_views.q4_weekday_viewers import make_plot_q4
from final_views.q5_seasonal_pattern import make_plot_q5

This notebook describes the celaning, data expolarating steps justification of the chosen visualizations and initial visualizations considered

**Authors**:
- Adrià Espinoza
- Biel Manté

# Data processing and exploration

## Data loading and new variables (cambiar)

We first load the data and perform the following steps:

1. Create a new variable, `day_aired`, from `original_air_date` (used to answer Q4).
2. Keep only the dataframe columns relevant to our analysis:
   - `season`
   - `number_in_season`
   - `number_in_series`
   - `original_air_date`
   - `day_aired`
   - `imdb_rating`
   - `us_viewers_in_millions`
   - `title`
3. Exclude columns that are not needed to answer our questions:
   - `id` (for our purposes, it is equivalent to `number_in_series`)
   - `image_url`
   - `imdb_votes`
   - `original_air_year` (can be derived from `original_air_date`)
   - `production_code`
   - `video_url`
   - `views`
4. Add a new binary column indicating whether an episode is the last one in its season. This variable was used in draft visualizations, but not in the final selected views.

In [20]:
path = "../data/simpsons_episodes.csv"
df = create_clean_dataframe(path)

## Check Null values and data types

We perform basic descriptive analysis to study the null values our dataset contains, as well as the data types of its variables

In [21]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   season                  600 non-null    int64         
 1   number_in_season        600 non-null    int64         
 2   number_in_series        600 non-null    int64         
 3   original_air_date       600 non-null    datetime64[ns]
 4   day_aired               600 non-null    object        
 5   imdb_rating             597 non-null    float64       
 6   us_viewers_in_millions  594 non-null    float64       
 7   title                   600 non-null    object        
 8   is_last_episode         600 non-null    int64         
dtypes: datetime64[ns](1), float64(2), int64(4), object(2)
memory usage: 42.3+ KB
None


We see that the dataset contains a small number of missing values. `imdb_rating` has 3 missing entries, `us_viewers_in_millions` has 6 missing values. We will decide in future sections how to deal with them


## Check Unique values

In [22]:
counts_df = df.groupby("season").size().reset_index(name="episode_count").sort_values("season")
print(counts_df)
print("---" * 10)
print(counts_df[counts_df["episode_count"] == counts_df["episode_count"].max()])
print("---" * 10)
print(counts_df[counts_df["episode_count"] == counts_df["episode_count"].min()])

    season  episode_count
0        1             13
1        2             22
2        3             24
3        4             22
4        5             22
5        6             25
6        7             25
7        8             25
8        9             25
9       10             23
10      11             22
11      12             21
12      13             22
13      14             22
14      15             22
15      16             21
16      17             22
17      18             22
18      19             20
19      20             21
20      21             23
21      22             22
22      23             22
23      24             22
24      25             22
25      26             22
26      27             22
27      28              4
------------------------------
   season  episode_count
5       6             25
6       7             25
7       8             25
8       9             25
------------------------------
    season  episode_count
27      28              4


In [23]:
print(df["day_aired"].value_counts())

day_aired
Sunday       506
Thursday      89
Tuesday        2
Wednesday      2
Friday         1
Name: count, dtype: int64


Our findings analyzing the unique values of the variables where it made sense to do so are:
- There are 28 seasons, and the maximum number of episodes in a season is 25, which is consistent with typical TV season structures. Seasons 6 to 9 share this maximum number
- Season 28 has the minimun number of episodes with only 4. This is extreamly different compared to the rest of the seasons 
- The `original_air_date` column contains 591 unique dates out of 600 episodes, meaning that 9 episodes share the same air date.
- The `day_aired` variable has only 5 unique values and is highly unbalanced: most episodes aired on Sunday (506), followed by Thursday (89), while Tuesday and Wednesday appear only twice each and Friday only once.

# Outlier Detection

An outlier detection analysis was performed using the IQR method on the numerical variables `imdb_rating` and `us_viewers_in_millions`. The results show a very small number of outliers in `imdb_rating` (2) and a slightly higher number in `us_viewers_in_millions` (9).

The 2 outliers for `imdb_rating` belong to very low ratings. This are:
* The 11th episode of season 9
* The 22th episode of season 11

In [24]:
outliers_imdb_rating = count_outliers_iqr(df, "imdb_rating")
lb = outliers_imdb_rating[2]
ub = outliers_imdb_rating[3]
outliers_imdb_rating[0][['season','original_air_date','day_aired','imdb_rating','us_viewers_in_millions','title']]

,season,original_air_date,day_aired,imdb_rating,us_viewers_in_millions,title
71,9,1998-01-04,Sunday,5.1,8.90,"All Singing, All Dancing"
193,23,2012-05-20,Sunday,4.5,4.82,Lisa Goes Gaga


Since there aren't many outliers and the rating scale (0–10) isn't particularly large, keeping them won't significantly impact the distribution or future visualizations. So we'll keep them as they are.

Let's visualize the distribution of ratings and see where these outliers fall:

In [25]:
hist = alt.Chart(df).mark_bar().encode(
    x=alt.X("imdb_rating:Q", bin=alt.Bin(maxbins=30), title="IMDb rating"),
    y=alt.Y("count():Q", title="Episodes")
)

bounds_df = pd.DataFrame({"bound": [lb, ub]})
bounds = alt.Chart(bounds_df).mark_rule(strokeDash=[4, 4]).encode(
    x="bound:Q"
 )

chart = (hist + bounds).properties(
    title="Histogram of IMDb Ratings with outlier bounds"
 )
chart

alt.LayerChart(...)

We find 9 outliers on the variable us_viewers_in_millions, all of them belonging to early seasons of the show and larger audiences
The Simpsons was transmitted right before the news programm in Fox News. Some meaningfull events happened on some of these days which could have made people want to watch the news more and indireclty boost the episodes audince (this is just observational and does not prove anyti).

Some of these special events were:
* On March 25, 1990, a tragic arson attack at the Happy Land social club in the Bronx, New York City, killed 87 people
* On April 29, 1990, the US space shuttle Discovery (STS-31) returned to Earth after deploying the Hubble Space Telescope. Major events included the LA Lakers defeating the Houston Rockets to set a record, Nirvana playing a legendary show in Washington, D.C., and Dan Quisenberry announcing his MLB retirement. 
* On December 3, 1992 The world's very first text was sent British engineer Neil Papworth sent the message “Merry Christmas” from his computer to the mobile phone of Vodafone director Richard Jarvis.

In [26]:
outliers_viewers = count_outliers_iqr(df, "us_viewers_in_millions")
lb = outliers_viewers[2]
ub = outliers_viewers[3]
outliers_viewers[0][['season','original_air_date','day_aired','imdb_rating','us_viewers_in_millions','title']].sort_values('original_air_date')

,season,original_air_date,day_aired,imdb_rating,us_viewers_in_millions,title
26,1,1990-02-18,Sunday,7.9,27.6,The Call of the Simpsons
245,1,1990-02-25,Sunday,7.7,28.0,The Telltale Head
217,1,1990-03-18,Sunday,7.5,33.5,Life on the Fast Lane
0,1,1990-03-25,Sunday,7.4,30.3,Homer's Night Out
218,1,1990-04-15,Sunday,7.8,31.2,The Crepes of Wrath
1,1,1990-04-29,Sunday,8.3,30.4,Krusty Gets Busted
2,2,1990-10-11,Thursday,8.2,33.6,"Bart Gets an ""F"""
220,2,1990-10-18,Thursday,8.3,29.9,Simpson and Delilah
24,4,1992-12-03,Thursday,8.5,28.6,Lisa's First Word


dir q no els elinimen?

## Dealing with null values

From previous sections, we found 6 missing values in `us_viewers_in_millions`. Three of them come from season 28, which only has 4 episodes.

Keeping a season with just 4 episodes does not really help us identify trends compared with the other seasons, which contain many more episodes. It could also create confusion in the visualizations, for example making users think the dataset is incomplete. For this reason, we decided to remove episodes from season 28 and keep seasons 1-27, which are enough to answer the questions.

By removing season 28, we eliminated all missing values in `imdb_rating` and half of the missing values in `us_viewers_in_millions`.

In [27]:
df_clean = df[df['season'] < 28]

To handle the 3 missing values that still remain in `us_viewers_in_millions`, we found a second dataset with a very similar structure to the provided one and matching values for `imdb_rating` and `us_viewers_in_millions`. We  decided to plug in the missing values in our dataset using the equivalent entries from this second source.

This second source dataset can be found here: https://www.kaggle.com/datasets/jonbown/simpsons-episodes-2016?select=simpsons_episodes.csv

In [28]:
dataset2 = pd.read_csv('../data/simpsons_episodes_second_source.csv')

In [29]:
df_clean = df_clean.merge(dataset2[['season', 'number_in_season', 'us_viewers_in_millions']], on=['season', 'number_in_season'], how='left', suffixes=('', '_from_ds2'))
df_clean['us_viewers_in_millions'] = df_clean['us_viewers_in_millions'].fillna(df_clean['us_viewers_in_millions_from_ds2'])
df_clean.drop(columns=['us_viewers_in_millions_from_ds2'], inplace=True)

After this step, we have completed all the preprocessing steps and we can save our clean dataset 

In [30]:
df_clean.to_csv('../data/simpsons_episodes_cleaned.csv', index=False)

# Questions justification

## Q1 - How have the ratings evolved over time?
To answer this question, we chose a heatmap. This chart allows us to display the rating of every episode while making it easy to see how ratings evolve across seasons and within each season.

We use season number on the X-axis and episode number within season on the Y-axis. We also place the X-axis labels at the top to improve readability, so users can find the first episode of each season more quickly without having to make long visual jumps. In addition, we added white lines between tiles to create clear visual cells and separate ratings.

For color, we selected a red-yellow-green scale. This follows common visual intuition (red = worse, green = better) and makes interpretation faster.  we ensured the palette was coloblind-friendly, and having a colorblind member in our group allowed us to test this. Using `scale = alt.Scale(range=['red', 'yellow', 'green'])` made low-rated episodes harder to distinguish, while switching to `scheme='redyellowgreen'` made distinctions much easier.

Finally, we removed the gradient legend in the final view because the color meaning is already intuitive. This saves space and makes the rating labels inside the cells easier to read.

In [36]:
make_plot_q1()


alt.LayerChart(...)

## Q2 - How have the viewers evolved over time?

To show how viewership evolved over time, we initially thought a line chart would be the best fit. Using time on the X-axis and the number of viewers (in millions) on the Y-axis, along with season separators, makes it easy to track the show’s growth and link specific peaks to their respective seasons.

Plotting all 596 episodes created a problem: the chart became too wide for the window, and the individual lines were so close together that the visualization looked cluttered. To fix this, we decided to plot the seasonal average as a solid line and display individual episodes as points. We lowered the opacity of the points to avoid oversaturating the plot while still havinng a way to see the "spread" of the data.

Even though we lose some detail by focusing on the mean, it highlights the overall trend accurately without being misleading. We kept the individual points so that the full detail for every season is still represented.

For the colors, we chose a strong blue for the seasonal average and a lighter cyan for the individual episodes. We used related colors to show that these measures are connected but distinct. The meaning of these colors is visible on the plot legend. 

Moving forward, we’ll stick to this logic for all our charts: bold blue for aggregated data and lighter tones for episode-specific values.

In [37]:
make_plot_q2()

alt.LayerChart(...)

## Q3 - Is there a correlation between the gradings and the viewers?


This question would be answered just by showing a number, the correlation coefficient. Since we are in a Data Visualization subject we aim to go straight further and provide some visualization that make understand the coefficient. So we decided to go for the Line Chart with a Dual Axis. Since we know that the numer of viewers are related to the ratings we decided to plot them into a Dual Axis making evident the correlation between them. X axis the time (the number of the season), the Y-axis the number of viewers and the ratings.


In this visualization we have only considered the averages across the different seasons because plotting the points would overlap. 

We use the nonparametric version of the peerson coeficient since the two varaibles do not share the same scale.

Finally, according to the color that we have used in the previous chart we have decided to be consistent and paint the viewers with the same color. For rating we have used a different color easy to differenciate for coloblind people.

(PARAPHRASE IT)

### GEMINI ANSWER

While this question could be answered simply by displaying a correlation coefficient, our goal is to visually demonstrate the relationship. Therefore, we designed a dual-axis line chart to make the correlation visually evident over time.

We mapped the progression of time (Seasons) on the shared X-axis. The left Y-axis represents US viewers (in millions), and the right Y-axis shows the IMDb ratings. To prevent severe overlapping and visual clutter from over 600 episodes, we aggregated the data to plot only the seasonal averages. This clearly highlights the macro-trend of both variables falling and rising together.

To statistically support the chart, we calculated Spearman’s rank correlation coefficient, which is ideal for this data as it does not assume a perfectly linear distribution.

Finally, to maintain design consistency across the dashboard, we plotted the viewership line in the same blue used in our previous charts. For the IMDb ratings, we selected a highly contrasting, colorblind-friendly color (orange)  to ensure all users can easily distinguish the two axes.

## Q4 - Are the number of viewers for the episodes related to the weekday they were aired?

The first thing we decided to adress this question was that we shoul focus it to only study the days with a reasonable number of episodes, so we restricted it to Sunday and Thursday.

The fist plot we tried was a jitter plot indicating the distribution, but it didn't convince us. We've also tested different superposed histograms and density plots. The Y-axis of this were normalized since there were way more epsiodes aired on Sunday. 

All these visualizations suggested that Thursday episodes had more viewership, but we realized that all the Thursday episodes belonged to seasons 2 through 5. Since we already know the show had much higher ratings in its earlier years, concluding that Thursday is a "better" day could be misleading since the difference could come from the season, not the day of the week.

To address this, we transformed the viewership data using differentiation to remove the trend, a common technique for time series data. We then compared these detrended values using overlapping density plots to see the actual impact of the airing day, to fins out that the distributions were not so different, sharing a large area of the distribution.

Since this plot encodes different information, we chose purple and pink for the weekdays to keep them distinct from the blue viewership tones and the red-yellow-green ratings used in other plots. They are also colorblind-friendly, the overlapping areas of the density plot are easy to tell distinguish.

In [39]:
make_plot_q4().properties(width=600, height=400)

alt.LayerChart(...)

## Q5 - Do the seasons’ number of viewers present any relevant pattern?

We approached this question by looking for patterns within individual seasons, for example, checking if premieres and finales consistently attract more viewers than mid-season episodes. 

Our initial idea was to create separate views for each season to observe these trends. However, the high number of seasons made this approach space-inefficient and made it difficult to compare them side-by-side, as not all charts could be horizontally aligned. We decided to continue a different alternatives.

We wanted a way to be able to represnet seaons with different lenghts in a shared X-axis, so we introduced a new variable that normalized the episode number by the number of episodes of the season. Values close to 0 belong to early espidoes and the ones close to 1 to the later ones.  In the y-axis we displayed the number of viewers.

We also know that later seasons had more views than ealier ones, and since they are mixed in similar points of the X-axis, it was difficult to spot patterns. We addressed this by transforming viewers into a relative measure: the percentage of a season’s total viewers.

Finally, to make the general trend easier to see, we added a tendency line calculated from the average of ten quantiles across the normalized X-axis.

In [40]:
make_plot_q5()

alt.LayerChart(...)

## Final Visualization

The dashboard is structured to guide the user's analytical flow from top-left to bottom-right, fully optimized to fit cleanly on a single computer screen without excessive scrolling. To facilitate direct temporal comparisons, visualizations that share a time-based X-axis are strategically aligned. 

We also implemented a strict semantic color strategy to reduce cognitive load across the dashboard: blue consistently represents viewership metrics, while orange denotes IMDb ratings, which allows for clear differentiation in the dual-axis correlation chart. We separated our palettes by data type, using distinct categorical colors for day-of-the-week encodings so they do not conflict with the diverging color schemes used in the ratings heatmap. 

Finally, every palette was tested for color-blind accessibility, ensuring the final visual narrative is both aesthetically cohesive and universally readable.